# SRCNN ×3 — 딥러닝 초해상화의 시작

흐린 위성사진(10 m) → 3배 선명하게(3.33 m).

**2014년, 이 분야 최초의 CNN 이다.** conv 3장이 전부고 파라미터는 57.3K —
HAT 의 **1/363** 이다. 앞선 모델들과 규약이 두 가지 다르다.

- **모델 안에 업샘플이 없다.** 먼저 bicubic 으로 3배 키운 뒤 그 흐린 영상을 다듬는다.
- **밝기(Y) 채널 하나만 본다.** 색은 bicubic 확대본을 그대로 쓴다.

| | 구조 | 손실 | 파라미터 |
|---|---|---|---|
| **SRCNN** | **conv 3장** | **MSE** | **57.3K** |
| EDSR | CNN (residual) | L1 | 1.55M |
| SRGAN | CNN + GAN | MSE + VGG + 적대적 | 0.77M |
| ESRGAN | CNN (RRDB) + GAN | L1 + VGG + RaGAN | 5.91M |
| SwinIR | Transformer (Swin) | L1 | 11.94M |
| HAT | Transformer (Swin + 채널·중첩 어텐션) | L1 | 20.81M |

## 1. 데이터

In [ ]:
import sys, urllib.request

LIB = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main/lib'
for m in ['sr_utils.py', 'srcnn_arch.py', 'srcnn_imgproc.py', 'srcnn_models.py']:
    urllib.request.urlretrieve(f'{LIB}/{m}', m)
    sys.modules.pop(m[:-3], None)     # 이미 불러온 옛 모듈이 남아 있으면 비운다

from sr_utils import *

show_data()        # validation 2패치 + test 2구역

## 2. 훈련

**코드가 도는지 확인하는 용도다.** 16장으로 1 epoch 만 돌린다.
아래 결과는 전체 데이터로 학습해둔 가중치를 쓴다.

모델이 워낙 작아 GPU 가 없어도 돈다. `srcnn_pairs` 가 LR 을 bicubic 으로 3배 키운 뒤
Y 채널만 뽑아 입력을 만들고, 목표는 HR 의 Y 채널이다.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from srcnn_models import build_srcnn, srcnn_pairs

N_TRAIN, EPOCHS, BATCH = 16, 1, 4
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

lo, hi = zip(*[pair('training', s) for s in list_split('training')[:N_TRAIN]])
x, t = srcnn_pairs(lo, hi)        # 입력: bicubic 으로 키운 LR 의 Y / 목표: HR 의 Y
loader = DataLoader(TensorDataset(x, t), batch_size=BATCH, shuffle=True)

net = build_srcnn().to(dev).train()
opt = torch.optim.Adam(net.parameters(), 2e-4, betas=(0.9, 0.99))
crit = nn.MSELoss()

print(f'SRCNN  {sum(p.numel() for p in net.parameters())/1e3:.1f}K')
for ep in range(1, EPOCHS + 1):
    tot = 0.0
    for xb, tb in loader:
        loss = crit(net(xb.to(dev)), tb.to(dev))
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item()
    print(f'epoch {ep}/{EPOCHS}   MSE {tot/len(loader):.6f}')

## 3. 학습 로그

전체 데이터로 100 epoch 돌린 기록이다.

In [ ]:
import pandas as pd

MODEL = f'{BASE}/models/06_srcnn_x3'
e = pd.read_csv(fetch(f'{MODEL}/statistics/train_results.csv', 'log.csv'), index_col=0)

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(e.index, e.mse, color='#2f6f9f', lw=1.6)
ax[0].set_yscale('log'); ax[0].set_title('Training MSE loss'); ax[0].set_ylabel('MSE')

ax[1].plot(e.index, e.PSNR, color='#4f9d69', lw=1.6)
ax[1].set_title('Validation PSNR during training'); ax[1].set_ylabel('dB')

ax[2].step(e.index, e.lr, color='#c96a5b', lw=1.6, where='post')
ax[2].set_yscale('log'); ax[2].set_title('Learning rate (halved 3 times)')
for a in ax: a.set_xlabel('epoch'); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f'MSE  {e.mse.iloc[0]:.6f} -> {e.mse.iloc[-1]:.6f}')

## 4. 결과

In [ ]:
from srcnn_models import load_srcnn, srcnn_upscale

net = load_srcnn(fetch(f'{MODEL}/checkpoints/srcnn_x3.pth', 'srcnn_x3.pth'))
print(f'SRCNN  {sum(p.numel() for p in net.parameters())/1e3:.1f}K')

upscale = lambda lr: srcnn_upscale(net, lr)

show_results(upscale, 'SRCNN')      # center=(x, y) 로 확대 위치 지정 (아래 라벨에 현재 값)

## 5. 평가

In [ ]:
rows = compare(upscale, label='SRCNN')

## 6. 최종 테스트 — 인천

정답이 없는 실제 Sentinel-2 촬영본이다. 점수는 못 내고 눈으로 확인한다.

In [ ]:
show_test(upscale, 'SRCNN')         # center=(x, y), size=110 으로 조절